# Reproduce Pair Tables

Compare the 6 CSVs in `data/processed/` (original) against the reproduced versions
in `outputs/reproduce-pairs/`.

**Columns compared per file type:**

| File | Compared | Skipped (derived / order-dependent) |
|------|----------|-------------------------------------|
| `*_pairs.csv` | mutant_profile1/2, concentration, median_diff | t_stat, p_value, profile\*_rep\* |
| `*_global_fitness.csv` | mutant_profile, fitness, error, normalized_fitness | auc1/2/3 |
| `*_global_pairs.csv` | mutant_profile1/2, fitness_diff | — |

In [73]:
from pathlib import Path

import polars as pl

BASE = Path("../../")
ORIGINAL = BASE / "data" / "processed"
REPRODUCED = Path("outputs") / "reproduce-pairs"

In [74]:
def canonicalize_pairs(df: pl.DataFrame) -> pl.DataFrame:
    """Canonicalize pair rows so mutant_profile1 <= mutant_profile2 lexicographically.

    When swapping, negates t_stat/median_diff/fitness_diff and swaps replicate columns.
    """
    needs_swap = df["mutant_profile1"] > df["mutant_profile2"]

    new_p1 = pl.when(needs_swap).then(pl.col("mutant_profile2")).otherwise(pl.col("mutant_profile1")).alias("mutant_profile1")
    new_p2 = pl.when(needs_swap).then(pl.col("mutant_profile1")).otherwise(pl.col("mutant_profile2")).alias("mutant_profile2")
    select_exprs = [new_p1, new_p2]

    for col in df.columns:
        if col in ("mutant_profile1", "mutant_profile2"):
            continue
        if col in ("t_stat", "median_diff", "fitness_diff"):
            select_exprs.append(
                pl.when(needs_swap).then(-pl.col(col)).otherwise(pl.col(col)).alias(col)
            )
        elif col.startswith("profile1_"):
            col2 = col.replace("profile1_", "profile2_")
            select_exprs.append(
                pl.when(needs_swap).then(pl.col(col2)).otherwise(pl.col(col)).alias(col)
            )
        elif col.startswith("profile2_"):
            col1 = col.replace("profile2_", "profile1_")
            select_exprs.append(
                pl.when(needs_swap).then(pl.col(col1)).otherwise(pl.col(col)).alias(col)
            )
        else:
            select_exprs.append(pl.col(col))

    return df.select(select_exprs)


def compare_csvs(
    name: str,
    original_dir: Path,
    reproduced_dir: Path,
    sort_cols: list[str],
    atol: float = 1e-6,
    has_pairs: bool = False,
    skip_cols: set[str] | None = None,
) -> tuple[dict, pl.DataFrame, pl.DataFrame]:
    """Compare two CSV files and report differences.

    Args:
        name: CSV filename.
        original_dir: Directory containing the original CSV.
        reproduced_dir: Directory containing the reproduced CSV.
        sort_cols: Columns to sort by before comparison.
        atol: Absolute tolerance for float comparison.
        has_pairs: If True, canonicalize mutant_profile1/2 ordering before sorting.
        skip_cols: Column names to exclude from comparison.

    Returns:
        Tuple of (summary dict, sorted original df, sorted reproduced df).
    """
    skip = skip_cols or set()

    print(f"\n{'=' * 60}")
    print(f"Comparing: {name}")
    print(f"{'=' * 60}")

    orig = pl.read_csv(original_dir / name)
    repro = pl.read_csv(reproduced_dir / name)

    result = {"name": name, "pass": True, "issues": []}

    # Shape
    if orig.shape == repro.shape:
        print(f"  Shape: {orig.shape} ✓")
    else:
        print(f"  Shape MISMATCH: original={orig.shape}, reproduced={repro.shape}")
        result["pass"] = False
        result["issues"].append(f"shape {orig.shape} vs {repro.shape}")

    # Columns
    if orig.columns == repro.columns:
        print(f"  Columns: match ✓")
    else:
        only_orig = set(orig.columns) - set(repro.columns)
        only_repro = set(repro.columns) - set(orig.columns)
        print(f"  Columns MISMATCH: only_in_original={only_orig}, only_in_reproduced={only_repro}")
        result["pass"] = False
        result["issues"].append("column names differ")
        return result, orig, repro

    # Canonicalize pair ordering
    if has_pairs:
        orig = canonicalize_pairs(orig)
        repro = canonicalize_pairs(repro)

    # Sort
    orig = orig.sort(sort_cols)
    repro = repro.sort(sort_cols)

    if skip:
        print(f"  Skipping: {sorted(skip)}")

    # Per-column comparison
    for col in orig.columns:
        if col in skip:
            continue
        dt = orig.schema[col]
        if dt == pl.Utf8:
            n_diff = (orig[col] != repro[col]).sum()
            if n_diff == 0:
                print(f"  {col}: exact match ✓")
            else:
                print(f"  {col}: {n_diff} rows differ ✗")
                result["pass"] = False
                result["issues"].append(f"{col}: {n_diff} string diffs")
        elif dt in (pl.Float32, pl.Float64, pl.Int32, pl.Int64):
            diff = (orig[col].cast(pl.Float64) - repro[col].cast(pl.Float64)).abs()
            max_diff = diff.max()
            n_over_tol = (diff > atol).sum()
            if n_over_tol == 0:
                print(f"  {col}: max_diff={max_diff:.2e} ✓")
            else:
                print(f"  {col}: max_diff={max_diff:.2e}, {n_over_tol} rows > atol ✗")
                result["pass"] = False
                result["issues"].append(f"{col}: {n_over_tol} diffs > {atol}")
        else:
            print(f"  {col} (dtype={dt}): skipped")

    status = "PASS ✓" if result["pass"] else "FAIL ✗"
    print(f"\n  Result: {status}")
    return result, orig, repro

## Concentration-specific pair tables

In [75]:
PAIR_SORT = ["mutant_profile1", "mutant_profile2", "concentration"]
PAIR_SKIP = {
    "t_stat", "p_value",
    "profile1_rep1", "profile1_rep2", "profile1_rep3",
    "profile2_rep1", "profile2_rep2", "profile2_rep3",
}

r_amp_pairs, orig_amp_pairs, repro_amp_pairs = compare_csvs(
    "amp_pairs.csv", ORIGINAL, REPRODUCED,
    sort_cols=PAIR_SORT, has_pairs=True, skip_cols=PAIR_SKIP,
)


Comparing: amp_pairs.csv
  Shape: (2985660, 12) ✓
  Columns: match ✓
  Skipping: ['p_value', 'profile1_rep1', 'profile1_rep2', 'profile1_rep3', 'profile2_rep1', 'profile2_rep2', 'profile2_rep3', 't_stat']
  mutant_profile1: exact match ✓
  mutant_profile2: exact match ✓
  concentration: max_diff=0.00e+00 ✓
  median_diff: max_diff=0.00e+00 ✓

  Result: PASS ✓


In [76]:
r_azt_pairs, orig_azt_pairs, repro_azt_pairs = compare_csvs(
    "azt_pairs.csv", ORIGINAL, REPRODUCED,
    sort_cols=PAIR_SORT, has_pairs=True, skip_cols=PAIR_SKIP,
)


Comparing: azt_pairs.csv
  Shape: (3980880, 12) ✓
  Columns: match ✓
  Skipping: ['p_value', 'profile1_rep1', 'profile1_rep2', 'profile1_rep3', 'profile2_rep1', 'profile2_rep2', 'profile2_rep3', 't_stat']
  mutant_profile1: exact match ✓
  mutant_profile2: exact match ✓
  concentration: max_diff=0.00e+00 ✓
  median_diff: max_diff=0.00e+00 ✓

  Result: PASS ✓


## Global fitness tables

In [77]:
FITNESS_SORT = ["mutant_profile"]
FITNESS_SKIP = {"auc1", "auc2", "auc3"}

r_amp_gf, orig_amp_gf, repro_amp_gf = compare_csvs(
    "amp_global_fitness.csv", ORIGINAL, REPRODUCED,
    sort_cols=FITNESS_SORT, skip_cols=FITNESS_SKIP,
)


Comparing: amp_global_fitness.csv
  Shape: (55294, 7) ✓
  Columns: match ✓
  Skipping: ['auc1', 'auc2', 'auc3']
  mutant_profile: exact match ✓
  fitness: max_diff=2.66e-15 ✓
  error: max_diff=8.19e-16 ✓
  normalized_fitness: max_diff=8.88e-16 ✓

  Result: PASS ✓


In [78]:
r_azt_gf, orig_azt_gf, repro_azt_gf = compare_csvs(
    "azt_global_fitness.csv", ORIGINAL, REPRODUCED,
    sort_cols=FITNESS_SORT, skip_cols=FITNESS_SKIP,
)


Comparing: azt_global_fitness.csv
  Shape: (55294, 7) ✓
  Columns: match ✓
  Skipping: ['auc1', 'auc2', 'auc3']
  mutant_profile: exact match ✓
  fitness: max_diff=1.78e-15 ✓
  error: max_diff=7.77e-16 ✓
  normalized_fitness: max_diff=3.55e-15 ✓

  Result: PASS ✓


## Global pair tables

In [79]:
GLOBAL_PAIR_SORT = ["mutant_profile1", "mutant_profile2"]

r_amp_gp, orig_amp_gp, repro_amp_gp = compare_csvs(
    "amp_global_pairs.csv", ORIGINAL, REPRODUCED,
    sort_cols=GLOBAL_PAIR_SORT, has_pairs=True,
)


Comparing: amp_global_pairs.csv
  Shape: (497610, 3) ✓
  Columns: match ✓
  mutant_profile1: exact match ✓
  mutant_profile2: exact match ✓
  fitness_diff: max_diff=3.55e-15 ✓

  Result: PASS ✓


In [80]:
r_azt_gp, orig_azt_gp, repro_azt_gp = compare_csvs(
    "azt_global_pairs.csv", ORIGINAL, REPRODUCED,
    sort_cols=GLOBAL_PAIR_SORT, has_pairs=True,
)


Comparing: azt_global_pairs.csv
  Shape: (497610, 3) ✓
  Columns: match ✓
  mutant_profile1: exact match ✓
  mutant_profile2: exact match ✓
  fitness_diff: max_diff=3.55e-15 ✓

  Result: PASS ✓


## Summary

In [81]:
all_results = [r_amp_pairs, r_azt_pairs, r_amp_gf, r_azt_gf, r_amp_gp, r_azt_gp]

print("\n" + "=" * 60)
print("OVERALL SUMMARY")
print("=" * 60)
for r in all_results:
    status = "PASS ✓" if r["pass"] else "FAIL ✗"
    issues = ", ".join(r["issues"]) if r["issues"] else ""
    suffix = f"  ({issues})" if issues else ""
    print(f"  {r['name']:30s} {status}{suffix}")

n_pass = sum(1 for r in all_results if r["pass"])
print(f"\n  {n_pass}/{len(all_results)} files match.")


OVERALL SUMMARY
  amp_pairs.csv                  PASS ✓
  azt_pairs.csv                  PASS ✓
  amp_global_fitness.csv         PASS ✓
  azt_global_fitness.csv         PASS ✓
  amp_global_pairs.csv           PASS ✓
  azt_global_pairs.csv           PASS ✓

  6/6 files match.
